In [2]:
import configparser

# Load config
config = configparser.ConfigParser()
config.read("config.ini")
weaviate_url = config["WEAVIATE"]["WEAVIATE_URL"]
weavitae_api_key = config["WEAVIATE"]["WEAVIATE_API"]
# print(weaviate_url)
# print('*' * 50)
# print(weavitae_api_key)

In [3]:
import weaviate

# Connect to Weaviate Cloud
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate.auth.AuthApiKey(weavitae_api_key)
)

# check if the connection is successful
client.is_ready()

True

In [17]:
client.collections.list_all()

{}

In [18]:
import pandas as pd
df = pd.read_csv(r'C:\Users\aakamaha\PycharmProjects\keysight\vanilla_rag\movie_metadata.csv')
df.head(5)

,director_name,genres,movie_title,language,country,title_year,imdb_score
0,James Cameron,Action|Adventure|Fantasy|Sci-Fi,Avatar,English,USA,2009,7.9
1,Gore Verbinski,Action|Adventure|Fantasy,Pirates of the Caribbean: At World's End,English,USA,2007,7.1
2,Sam Mendes,Action|Adventure|Thriller,Spectre,English,UK,2015,6.8
3,Christopher Nolan,Action|Thriller,The Dark Knight Rises,English,USA,2012,8.5
4,Andrew Stanton,Action|Adventure|Sci-Fi,John Carter,English,USA,2012,6.6


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4153 entries, 0 to 4152
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   director_name  4153 non-null   object 
 1   genres         4153 non-null   object 
 2   movie_title    4153 non-null   object 
 3   language       4152 non-null   object 
 4   country        4153 non-null   object 
 5   title_year     4153 non-null   int64  
 6   imdb_score     4153 non-null   float64
dtypes: float64(1), int64(1), object(5)
memory usage: 227.2+ KB


In [20]:
# Handle missing language value
df["language"] = df["language"].fillna("Unknown")

In [21]:
from weaviate.classes.config import Configure

# -----------------------------
# # delete collection if it exists
# -----------------------------

COLLECTION_NAME = 'Movie_Metadata'

if client.collections.exists(COLLECTION_NAME):
    print(f"Deleting existing collection: {COLLECTION_NAME}")
    client.collections.delete(COLLECTION_NAME)

# -----------------------------
# Create collection
# -----------------------------

client.collections.create(
    name=COLLECTION_NAME,

    vector_config=Configure.Vectors.text2vec_weaviate(
        model="Snowflake/snowflake-arctic-embed-l-v2.0",
        source_properties=[
            "movie_title",
            "director_name",
            "genres",
            "language",
            "country"
        ],
    ))

print(f"Successfully created collection: {COLLECTION_NAME}")    

Successfully created collection: Movie_Metadata


In [4]:
client.collections.get("Movie_Metadata").exists()


True

In [23]:
# -----------------------------
# Insert CSV records
# -----------------------------
records = df.to_dict(orient="records")

print(f"Inserting {len(records)} records...")

movie_collection = client.collections.use(COLLECTION_NAME)

with movie_collection.batch.fixed_size(batch_size=100) as batch:

    for record in records:

        batch.add_object(
            properties={
                "movie_title": record["movie_title"],
                "director_name": record["director_name"],
                "genres": record["genres"],
                "language": record["language"],
                "country": record["country"],
                "title_year": int(record["title_year"]),
                "imdb_score": float(record["imdb_score"]),
            }
        )

print("Ingestion completed.")

# -----------------------------
# Verify object count
# -----------------------------

result = movie_collection.aggregate.over_all(total_count=True)

print(f"Objects in Weaviate: {result.total_count}")


Inserting 4153 records...
Ingestion completed.
Objects in Weaviate: 4153


In [5]:
#access imported data from weaviate

movie_collection= client.collections.use("Movie_Metadata")
results = movie_collection.query.fetch_objects(limit=5)

for obj in results.objects:
    print(obj.properties)
    print("----")


{'language': 'English', 'imdb_score': 6.0, 'country': 'USA', 'movie_title': 'Raising Cain\xa0', 'title_year': 1992.0, 'director_name': 'Brian De Palma', 'genres': 'Crime|Drama|Thriller'}
----
{'language': 'English', 'imdb_score': 6.3, 'country': 'UK', 'movie_title': 'Transcendence\xa0', 'title_year': 2014.0, 'director_name': 'Wally Pfister', 'genres': 'Drama|Mystery|Romance|Sci-Fi|Thriller'}
----
{'language': 'English', 'imdb_score': 7.6, 'country': 'USA', 'movie_title': "It's a Mad, Mad, Mad, Mad World\xa0", 'title_year': 1963.0, 'director_name': 'Stanley Kramer', 'genres': 'Action|Adventure|Comedy|Crime'}
----
{'language': 'English', 'imdb_score': 5.8, 'country': 'USA', 'movie_title': 'Straw Dogs\xa0', 'title_year': 2011.0, 'director_name': 'Rod Lurie', 'genres': 'Action|Drama|Thriller'}
----
{'language': 'English', 'imdb_score': 6.8, 'country': 'USA', 'movie_title': 'Spun\xa0', 'title_year': 2002.0, 'genres': 'Comedy|Crime|Drama', 'director_name': 'Jonas Åkerlund'}
----


In [8]:
import json
query = "Action movies from 2012 to 2014"

movie_collection= client.collections.use("Movie_Metadata")

results = movie_collection.query.near_text(
        query=query,
        limit=5
    )

for obj in results.objects:
    print(json.dumps(obj.properties, indent=2))  # Inspect the results

{
  "language": "English",
  "imdb_score": 5.8,
  "country": "USA",
  "movie_title": "2012\u00a0",
  "title_year": 2009.0,
  "genres": "Action|Adventure|Sci-Fi",
  "director_name": "Roland Emmerich"
}
{
  "language": "English",
  "imdb_score": 7.4,
  "country": "USA",
  "movie_title": "13 Hours\u00a0",
  "title_year": 2016.0,
  "genres": "Action|Drama|Thriller|War",
  "director_name": "Michael Bay"
}
{
  "language": "English",
  "imdb_score": 6.0,
  "country": "USA",
  "movie_title": "Punisher: War Zone\u00a0",
  "title_year": 2008.0,
  "director_name": "Lexi Alexander",
  "genres": "Action|Crime|Drama|Thriller"
}
{
  "language": "English",
  "imdb_score": 3.7,
  "country": "USA",
  "movie_title": "Batman & Robin\u00a0",
  "title_year": 1997.0,
  "genres": "Action",
  "director_name": "Joel Schumacher"
}
{
  "language": "English",
  "imdb_score": 5.2,
  "country": "USA",
  "movie_title": "Action Jackson\u00a0",
  "title_year": 1988.0,
  "director_name": "Craig R. Baxley",
  "genres": "

In [25]:
client.close()